# Module 3 — Fonctions et organisation du code

> **Objectif du module** : Structurer son code en blocs réutilisables, gérer les erreurs proprement et exploiter les modules de la bibliothèque standard Python. À la fin de ce module, je suis capable d'écrire des scripts organisés et robustes pour automatiser des calculs de gestion.

> **Prérequis** : Avoir maîtrisé le Module 2 — listes, dictionnaires, conditions, boucles.

---

## 1. Créer et appeler une fonction

Une **fonction** est un bloc de code nommé qu'on peut appeler autant de fois qu'on veut. Elle permet d'éviter de répéter le même code et de le rendre lisible et maintenable.

Structure minimale :
```python
def nom_de_la_fonction(parametre1, parametre2):
    # corps de la fonction (indenté)
    return resultat
```

- `def` : mot-clé qui déclare la fonction
- Les **paramètres** sont des variables locales reçues à l'appel
- `return` renvoie une valeur — sans `return`, la fonction renvoie `None`

In [ ]:
# ── Exemple 1 : Première fonction — calcul de TVA ────────────────────────────

# Définition de la fonction
def calculer_ttc(montant_ht, taux_tva):    # 2 paramètres obligatoires
    """Calcule le montant TTC à partir du HT et du taux de TVA."""
    # Les triples guillemets sont la docstring : documentation de la fonction
    montant_ttc = montant_ht * (1 + taux_tva)
    return montant_ttc                      # renvoie le résultat

# Appel de la fonction — on peut la réutiliser autant de fois qu'on veut
prix1 = calculer_ttc(100, 0.20)    # → 120.0
prix2 = calculer_ttc(450, 0.20)    # → 540.0
prix3 = calculer_ttc(80, 0.055)    # → 84.4  (TVA réduite 5.5%)

print(f"100 HT → {prix1:.2f} € TTC")
print(f"450 HT → {prix2:.2f} € TTC")
print(f" 80 HT → {prix3:.2f} € TTC")

# Accéder à la documentation de la fonction
help(calculer_ttc)

In [ ]:
# ── Exemple 2 : Fonction sans return — effet de bord ────────────────────────
# Parfois une fonction affiche un résultat sans le retourner.
# Utile pour les fonctions d'affichage / reporting.

def afficher_fiche_client(client):
    """Affiche une fiche client formatée. Attend un dictionnaire client."""
    print(f"{'─' * 35}")
    print(f"  Client  : {client['nom']}")
    print(f"  Région  : {client['region']}")
    print(f"  CA      : {client['ca']:>12,} €")
    print(f"  Statut  : {'✓ Actif' if client['actif'] else '✗ Inactif'}")
    print(f"{'─' * 35}")
    # Pas de return → renvoie None implicitement

client = {"nom": "Dupont SA", "region": "Grand Est", "ca": 87500, "actif": True}
afficher_fiche_client(client)

---

## 2. Paramètres : valeurs par défaut et arguments nommés

On peut définir des **valeurs par défaut** pour les paramètres. Si l'appelant ne fournit pas la valeur, le défaut est utilisé. Cela rend les fonctions plus flexibles.

In [ ]:
# ── Exemple 3 : Valeurs par défaut ──────────────────────────────────────────

# taux_tva a une valeur par défaut de 0.20
# remise a une valeur par défaut de 0 (pas de remise)
def calculer_total(montant_ht, taux_tva=0.20, remise=0):
    """Calcule le total TTC après remise éventuelle."""
    montant_apres_remise = montant_ht * (1 - remise)
    return montant_apres_remise * (1 + taux_tva)

# Appel minimal : seul montant_ht est obligatoire
print(calculer_total(500))                      # → 600.0   (TVA 20%, sans remise)

# Appel avec un argument positionnel supplémentaire
print(calculer_total(500, 0.055))               # → 527.5   (TVA 5.5%, sans remise)

# Appel avec arguments nommés — l'ordre n'a plus d'importance
print(calculer_total(500, remise=0.10))         # → 540.0   (TVA 20%, remise 10%)
print(calculer_total(remise=0.15, montant_ht=500, taux_tva=0.20))  # → 510.0

---

## 3. Retourner plusieurs valeurs

Une fonction Python peut retourner **plusieurs valeurs** à la fois, sous forme de tuple. C'est très utile pour les fonctions d'analyse qui calculent plusieurs indicateurs en une passe.

In [ ]:
# ── Exemple 4 : Retourner plusieurs valeurs ──────────────────────────────────

def analyser_ventes(liste_montants):
    """Retourne les statistiques clés d'une liste de montants."""
    total   = sum(liste_montants)
    moyenne = total / len(liste_montants)
    minimum = min(liste_montants)
    maximum = max(liste_montants)
    return total, moyenne, minimum, maximum    # retour de 4 valeurs (tuple)

montants = [1200, 3400, 870, 2100, 4500, 980]

# Déballage (unpacking) : chaque valeur va dans sa variable
total, moyenne, mini, maxi = analyser_ventes(montants)

print(f"Total   : {total:,} €")
print(f"Moyenne : {moyenne:,.2f} €")
print(f"Mini    : {mini:,} €")
print(f"Maxi    : {maxi:,} €")

# On peut aussi récupérer uniquement ce dont on a besoin
total_seul, *_ = analyser_ventes(montants)    # *_ ignore le reste
print(f"\nJuste le total : {total_seul:,} €")

In [ ]:
# ── Exemple 5 : Retourner un dictionnaire de résultats ───────────────────────
# Souvent plus lisible que le tuple quand on a beaucoup de valeurs

def stats_portefeuille(clients):
    """Calcule les KPIs principaux d'un portefeuille clients."""
    liste_ca   = [c["ca"] for c in clients]
    actifs     = [c for c in clients if c["actif"]]
    en_retard  = [c for c in clients if c["retard"]]

    return {
        "nb_clients"  : len(clients),
        "nb_actifs"   : len(actifs),
        "nb_retards"  : len(en_retard),
        "ca_total"    : sum(liste_ca),
        "ca_moyen"    : sum(liste_ca) / len(liste_ca),
        "ca_max"      : max(liste_ca)
    }

clients = [
    {"nom": "Dupont SA",        "ca": 87500,  "actif": True,  "retard": False},
    {"nom": "Martin Logistics", "ca": 213000, "actif": True,  "retard": False},
    {"nom": "Petit & Fils",     "ca": 12800,  "actif": False, "retard": False},
    {"nom": "Moreau & Co",      "ca": 154000, "actif": True,  "retard": True}
]

kpis = stats_portefeuille(clients)

# On accède aux résultats par clé, c'est plus lisible qu'un tuple
print(f"Clients       : {kpis['nb_clients']}")
print(f"Actifs        : {kpis['nb_actifs']}")
print(f"En retard     : {kpis['nb_retards']}")
print(f"CA total      : {kpis['ca_total']:,} €")
print(f"CA moyen      : {kpis['ca_moyen']:,.2f} €")

---

## 4. Portée des variables (scope)

Une variable créée **à l'intérieur** d'une fonction est **locale** : elle n'existe que pendant l'exécution de la fonction et disparaît ensuite. Une variable créée **à l'extérieur** est **globale** et accessible partout.

C'est une règle fondamentale pour comprendre pourquoi une variable est parfois inaccessible.

In [ ]:
# ── Exemple 6 : Variable locale vs globale ───────────────────────────────────

taux_tva = 0.20    # variable GLOBALE — définie hors de toute fonction

def calculer_ttc(montant_ht):
    # taux_tva est accessible ici (lecture seule depuis le scope global)
    montant_ttc = montant_ht * (1 + taux_tva)    # montant_ttc est LOCALE
    return montant_ttc

print(calculer_ttc(500))    # → 600.0
print(taux_tva)             # → 0.20 (toujours accessible)

# montant_ttc n'existe pas en dehors de la fonction
# print(montant_ttc)        # → NameError : name 'montant_ttc' is not defined

print("\n--- Bonne pratique ---")
# Plutôt que d'utiliser une variable globale dans la fonction,
# il vaut mieux la passer en paramètre — le code est plus clair et testable

def calculer_ttc_v2(montant_ht, taux=0.20):    # taux en paramètre avec valeur par défaut
    return montant_ht * (1 + taux)

print(calculer_ttc_v2(500))          # → 600.0
print(calculer_ttc_v2(500, 0.055))   # → 527.5

---

## 5. Modules et imports

Python est livré avec une **bibliothèque standard** très riche : des modules prêts à l'emploi pour les maths, les dates, les fichiers, etc. En data analyse, les plus utiles sont `math`, `datetime`, `random` et `statistics`.

On les charge avec `import`.

In [ ]:
# ── Exemple 7 : Module math ──────────────────────────────────────────────────

import math

# Arrondir correctement
print(round(2.675, 2))          # → 2.67  (comportement parfois surprenant)
print(math.floor(2.9))          # → 2     (arrondi vers le bas)
print(math.ceil(2.1))           # → 3     (arrondi vers le haut)

# Racine carrée — utile pour les calculs statistiques
print(math.sqrt(144))           # → 12.0

# Logarithme — utile pour les taux de croissance
print(math.log(100, 10))        # → 2.0   (log base 10 de 100)

# Exemple data : taux de croissance annuel composé (CAGR)
ca_debut = 80000
ca_fin   = 125000
nb_ans   = 3
cagr = (ca_fin / ca_debut) ** (1 / nb_ans) - 1
print(f"\nCAGR sur {nb_ans} ans : {cagr:.2%}")    # → CAGR sur 3 ans : 16.08%

In [ ]:
# ── Exemple 8 : Module statistics ────────────────────────────────────────────

import statistics

ca_mensuel = [12500, 14200, 9800, 16100, 13750, 15400, 11200, 14800]

print(f"Moyenne   : {statistics.mean(ca_mensuel):,.2f} €")
print(f"Médiane   : {statistics.median(ca_mensuel):,.2f} €")
print(f"Écart-type: {statistics.stdev(ca_mensuel):,.2f} €")

# La médiane est plus robuste que la moyenne en présence de valeurs extrêmes
ca_avec_outlier = ca_mensuel + [250000]    # un mois exceptionnel
print(f"\nAvec outlier :")
print(f"  Moyenne : {statistics.mean(ca_avec_outlier):,.2f} €   ← très impactée")
print(f"  Médiane : {statistics.median(ca_avec_outlier):,.2f} €   ← stable")

In [ ]:
# ── Exemple 9 : Module datetime — incontournable en data analyse ─────────────

from datetime import date, datetime, timedelta
# from ... import ... : importer uniquement ce dont on a besoin

# Date d'aujourd'hui
aujourd_hui = date.today()
print(f"Aujourd'hui : {aujourd_hui}")                  # → 2025-01-15 (ex.)

# Créer une date précise
date_commande = date(2024, 11, 3)
print(f"Commande du : {date_commande}")

# Différence entre deux dates
delai = aujourd_hui - date_commande
print(f"Il y a {delai.days} jours")

# Ajouter / soustraire des jours avec timedelta
echeance = date_commande + timedelta(days=30)
print(f"Échéance paiement : {echeance}")

# Formater une date pour l'affichage
print(f"Formatted : {aujourd_hui.strftime('%d/%m/%Y')}")

# Parser une date depuis un texte (fréquent avec des CSV)
date_texte = "15/03/2024"
date_parsee = datetime.strptime(date_texte, "%d/%m/%Y").date()
print(f"Parsée : {date_parsee} — type : {type(date_parsee)}")

# Extraire le mois, l'année, le trimestre
print(f"Mois : {date_parsee.month} — Année : {date_parsee.year}")
trimestre = math.ceil(date_parsee.month / 3)
print(f"Trimestre : T{trimestre}")

In [ ]:
# ── Exemple 10 : Import avec alias — syntaxe qu'on retrouvera partout ────────

# Dans les modules suivants (Pandas, Matplotlib...), les imports se font avec alias
# Voici la convention standard :

import math as m              # alias court
print(m.sqrt(25))             # → 5.0

# Anticipation des modules du Module 4 :
# import pandas as pd          ← on écrira pd.DataFrame(), pd.read_csv()...
# import matplotlib.pyplot as plt  ← on écrira plt.plot(), plt.show()...
# import seaborn as sns        ← on écrira sns.barplot()...

---

## 6. Gestion des erreurs — try / except

En data analyse, les données sont rarement propres. Une valeur manquante, un texte là où on attend un nombre, un fichier introuvable... Sans gestion d'erreur, le script s'arrête brutalement. Avec `try / except`, on peut **anticiper** ces problèmes et y répondre proprement.

In [ ]:
# ── Exemple 11 : Syntaxe de base try / except ────────────────────────────────

# Sans gestion d'erreur — le script plante si la valeur n'est pas convertible
# valeur = int("abc")    # → ValueError: invalid literal for int()

# Avec try / except
def convertir_en_nombre(valeur_texte):
    """Convertit un texte en float. Retourne None si impossible."""
    try:
        # Python tente d'exécuter ce bloc
        return float(valeur_texte)
    except ValueError:
        # Si une ValueError se produit, on arrive ici — pas de crash
        print(f"  ⚠ Valeur non convertible : '{valeur_texte}'")
        return None

valeurs_brutes = ["1250.50", "3400", "N/A", "2100.00", "", "870.00"]

montants = []
for v in valeurs_brutes:
    resultat = convertir_en_nombre(v)
    if resultat is not None:      # on n'ajoute que les valeurs valides
        montants.append(resultat)

print(f"\nMontants valides : {montants}")
print(f"Total            : {sum(montants):,.2f} €")

In [ ]:
# ── Exemple 12 : Gérer plusieurs types d'erreurs ─────────────────────────────

def lire_valeur_client(clients, index, cle):
    """Lit une valeur dans une liste de clients de façon sécurisée."""
    try:
        valeur = clients[index][cle]      # peut échouer de deux façons :
        return valeur
    except IndexError:
        # L'index est hors de la liste
        print(f"  ⚠ Pas de client à l'index {index}")
        return None
    except KeyError:
        # La clé n'existe pas dans le dictionnaire
        print(f"  ⚠ La clé '{cle}' n'existe pas")
        return None

clients = [
    {"nom": "Dupont SA", "ca": 87500},
    {"nom": "Martin Logistics", "ca": 213000}
]

print(lire_valeur_client(clients, 0, "nom"))     # → Dupont SA
print(lire_valeur_client(clients, 5, "nom"))     # → None (index hors limites)
print(lire_valeur_client(clients, 0, "email"))   # → None (clé inexistante)

In [ ]:
# ── Exemple 13 : try / except / else / finally ───────────────────────────────
# Structure complète — utile pour les opérations sur fichiers (Module 6)

def diviser(a, b):
    """Division sécurisée avec journalisation."""
    try:
        resultat = a / b
    except ZeroDivisionError:
        print("  ⚠ Division par zéro impossible")
        return None
    else:
        # Ce bloc s'exécute UNIQUEMENT si aucune exception n'a eu lieu
        print(f"  ✓ Résultat : {resultat:.4f}")
        return resultat
    finally:
        # Ce bloc s'exécute TOUJOURS (erreur ou pas) — utile pour fermer des fichiers
        print("  → Opération terminée")

diviser(100, 4)
print()
diviser(100, 0)

---

## 7. Mettre tout ensemble — fonctions de reporting

Un exemple complet qui combine tout ce qu'on vient d'apprendre : fonctions, modules, gestion d'erreurs.

In [ ]:
# ── Exemple 14 : Mini-pipeline de traitement de données ──────────────────────

from datetime import date
import statistics

# ── Fonctions utilitaires ────────────────────────────────────────────────────

def nettoyer_montant(valeur):
    """Convertit une valeur en float. Retourne None si impossible."""
    try:
        return float(str(valeur).replace(",", ".").strip())
    except (ValueError, TypeError):
        return None

def segmenter_client(ca):
    """Retourne le segment commercial selon le CA annuel."""
    if ca >= 150000:
        return "Grand compte"
    elif ca >= 50000:
        return "Compte intermédiaire"
    elif ca >= 10000:
        return "PME"
    return "Petit compte"

def calculer_kpis(clients):
    """Calcule et retourne les KPIs du portefeuille."""
    ca_valides = []
    for c in clients:
        ca = nettoyer_montant(c.get("ca_brut"))    # .get() évite KeyError
        if ca is not None:
            c["ca"]      = ca
            c["segment"] = segmenter_client(ca)
            ca_valides.append(ca)
        else:
            c["ca"]      = 0
            c["segment"] = "Inconnu"

    if not ca_valides:
        return None

    return {
        "total"     : sum(ca_valides),
        "moyenne"   : statistics.mean(ca_valides),
        "mediane"   : statistics.median(ca_valides),
        "ecart_type": statistics.stdev(ca_valides) if len(ca_valides) > 1 else 0
    }

def afficher_rapport(clients, kpis):
    """Affiche le rapport complet du portefeuille."""
    print(f"{'═' * 50}")
    print(f"  RAPPORT PORTEFEUILLE — {date.today().strftime('%d/%m/%Y')}")
    print(f"{'═' * 50}")

    print(f"\n{'Nom':<22} {'CA':>12} {'Segment'}")
    print(f"{'─' * 50}")
    for c in sorted(clients, key=lambda x: x["ca"], reverse=True):
        print(f"{c['nom']:<22} {c['ca']:>12,.0f} €  {c['segment']}")

    print(f"\n{'─' * 50}")
    print(f"  CA total      : {kpis['total']:>12,.2f} €")
    print(f"  CA moyen      : {kpis['moyenne']:>12,.2f} €")
    print(f"  Médiane       : {kpis['mediane']:>12,.2f} €")
    print(f"  Écart-type    : {kpis['ecart_type']:>12,.2f} €")
    print(f"{'═' * 50}")

# ── Données brutes (comme si elles venaient d'un CSV) ───────────────────────
clients_bruts = [
    {"nom": "Dupont SA",          "ca_brut": "87500"},
    {"nom": "Martin Logistics",   "ca_brut": "213000"},
    {"nom": "Bernard Industries", "ca_brut": "N/A"},      # donnée corrompue
    {"nom": "Petit & Fils",       "ca_brut": "12800"},
    {"nom": "Blanc Industrie",    "ca_brut": "321000"},
    {"nom": "Simon Négoce",       "ca_brut": "8900"},
]

# ── Exécution du pipeline ────────────────────────────────────────────────────
kpis = calculer_kpis(clients_bruts)
afficher_rapport(clients_bruts, kpis)

---

## ✅ Ce que je dois maîtriser avant de passer au Module 4

- [ ] Écrire une fonction avec `def`, des paramètres et `return`
- [ ] Utiliser des valeurs par défaut et des arguments nommés
- [ ] Retourner plusieurs valeurs et les déballer avec l'unpacking
- [ ] Comprendre la différence entre variable locale et globale
- [ ] Importer et utiliser `math`, `statistics`, `datetime`
- [ ] Manipuler des dates : créer, calculer un écart, formater, parser
- [ ] Protéger un traitement avec `try / except`
- [ ] Combiner plusieurs fonctions dans un mini-pipeline de traitement

---

## 🏋️ Exercices pratiques — Contexte : calculs automatisés de CA et marges

---

### Dataset partagé — à exécuter en premier

In [ ]:
# ── Dataset partagé — exécuter cette cellule avant les exercices ─────────────

commandes = [
    {"id": "CMD-001", "client": "Dupont SA",          "date": "12/01/2024", "qte": 5,  "prix_ht": 120.0,  "prix_achat": 75.0},
    {"id": "CMD-002", "client": "Martin Logistics",   "date": "18/01/2024", "qte": 12, "prix_ht": 340.0,  "prix_achat": 210.0},
    {"id": "CMD-003", "client": "Dupont SA",          "date": "03/02/2024", "qte": 3,  "prix_ht": "N/A",  "prix_achat": 75.0},
    {"id": "CMD-004", "client": "Blanc Industrie",    "date": "14/02/2024", "qte": 8,  "prix_ht": 215.0,  "prix_achat": 140.0},
    {"id": "CMD-005", "client": "Simon Négoce",       "date": "29/02/2024", "qte": 2,  "prix_ht": 89.0,   "prix_achat": 52.0},
    {"id": "CMD-006", "client": "Martin Logistics",   "date": "07/03/2024", "qte": 20, "prix_ht": 340.0,  "prix_achat": 210.0},
    {"id": "CMD-007", "client": "Blanc Industrie",    "date": "22/03/2024", "qte": 6,  "prix_ht": 215.0,  "prix_achat": 140.0},
    {"id": "CMD-008", "client": "Dupont SA",          "date": "31/03/2024", "qte": 10, "prix_ht": 120.0,  "prix_achat": 75.0},
]

print(f"Dataset chargé : {len(commandes)} commandes")

---

### Exercice 1 — Fonctions de calcul commercial

Écrire les trois fonctions suivantes :

1. `calculer_ca_ht(qte, prix_ht)` → retourne `qte × prix_ht`
2. `calculer_marge(ca_ht, qte, prix_achat)` → retourne `ca_ht - (qte × prix_achat)`
3. `calculer_taux_marge(marge, ca_ht)` → retourne `marge / ca_ht` (en proportion)

Puis les tester sur la commande CMD-002 :
```
CMD-002 | Martin Logistics
CA HT        : 4,080.00 €
Marge        : 1,560.00 €
Taux de marge: 38.24%
```

In [ ]:
# Exercice 1 — Votre code ici


---

### Exercice 2 — Traitement robuste avec gestion d'erreurs

Certaines commandes ont un `prix_ht` invalide (`"N/A"`). Écrire une fonction `traiter_commande(cmd)` qui :
1. Tente de calculer le CA HT et la marge pour une commande
2. Retourne un dictionnaire avec `id`, `client`, `ca_ht`, `marge`, `valide` (bool)
3. Si le prix_ht n'est pas convertible, marque la commande comme invalide (`valide: False`, `ca_ht: 0`, `marge: 0`)

Puis appliquer cette fonction sur toutes les commandes et afficher un résumé :
```
CMD-001 ✓ CA :    600.00 €  Marge :   225.00 €
CMD-002 ✓ CA :  4,080.00 €  Marge : 1,560.00 €
CMD-003 ✗ Données invalides
...
```

In [ ]:
# Exercice 2 — Votre code ici


---

### Exercice 3 — Analyse temporelle

En utilisant le module `datetime` :
1. Écrire une fonction `extraire_mois(date_str)` qui parse une date au format `"jj/mm/aaaa"` et retourne le numéro du mois
2. Calculer le CA HT total par mois (en ignorant les commandes invalides)
3. Afficher le résultat trié par mois :
```
Mois 1 (Janvier) : 4,680.00 €
Mois 2 (Février) : ...
Mois 3 (Mars)    : ...
```

> 💡 **Indice** : les noms de mois peuvent être stockés dans une liste `["Janvier", "Février", ...]` et accédés par `noms_mois[num_mois - 1]`.

In [ ]:
# Exercice 3 — Votre code ici


---

### Exercice 4 — Rapport complet

Écrire une fonction `generer_rapport(commandes)` qui :
1. Traite toutes les commandes (en réutilisant les fonctions des exercices précédents)
2. Calcule le CA total, la marge totale et le taux de marge global
3. Identifie le client avec le plus grand CA
4. Compte les commandes valides et invalides
5. Affiche un rapport structuré

```
╔══════════════════════════════════════╗
║     RAPPORT COMMERCIAL T1 2024       ║
╠══════════════════════════════════════╣
║ Commandes traitées :    8            ║
║ Commandes valides  :    7            ║
║ CA total HT        : xx,xxx.xx €     ║
║ Marge totale       : xx,xxx.xx €     ║
║ Taux de marge      :    xx.xx%       ║
║ Top client         : xxxxx           ║
╚══════════════════════════════════════╝
```

In [ ]:
# Exercice 4 — Votre code ici


---

## 🔑 Corrections

Lire uniquement après avoir tenté les exercices.

In [ ]:
# ── CORRECTION Exercice 1 ────────────────────────────────────────────────────

def calculer_ca_ht(qte, prix_ht):
    """Retourne le CA HT : quantité × prix unitaire HT."""
    return qte * prix_ht

def calculer_marge(ca_ht, qte, prix_achat):
    """Retourne la marge brute : CA HT - coût d'achat total."""
    cout_achat = qte * prix_achat
    return ca_ht - cout_achat

def calculer_taux_marge(marge, ca_ht):
    """Retourne le taux de marge en proportion (0.38 = 38%)."""
    if ca_ht == 0:
        return 0
    return marge / ca_ht

# Test sur CMD-002
cmd = commandes[1]    # index 1 = CMD-002
ca    = calculer_ca_ht(cmd["qte"], float(cmd["prix_ht"]))
marge = calculer_marge(ca, cmd["qte"], cmd["prix_achat"])
taux  = calculer_taux_marge(marge, ca)

print(f"{cmd['id']} | {cmd['client']}")
print(f"CA HT        : {ca:,.2f} €")
print(f"Marge        : {marge:,.2f} €")
print(f"Taux de marge: {taux:.2%}")

In [ ]:
# ── CORRECTION Exercice 2 ────────────────────────────────────────────────────

def traiter_commande(cmd):
    """Traite une commande et retourne ses indicateurs. Gère les données invalides."""
    try:
        prix_ht = float(cmd["prix_ht"])    # peut lever ValueError si "N/A" etc.
        ca_ht   = calculer_ca_ht(cmd["qte"], prix_ht)
        marge   = calculer_marge(ca_ht, cmd["qte"], cmd["prix_achat"])
        return {"id": cmd["id"], "client": cmd["client"],
                "ca_ht": ca_ht, "marge": marge, "valide": True}
    except (ValueError, TypeError):
        return {"id": cmd["id"], "client": cmd["client"],
                "ca_ht": 0, "marge": 0, "valide": False}

resultats = [traiter_commande(c) for c in commandes]    # list comprehension

for r in resultats:
    if r["valide"]:
        print(f"{r['id']} ✓  CA : {r['ca_ht']:>10,.2f} €  Marge : {r['marge']:>8,.2f} €")
    else:
        print(f"{r['id']} ✗  Données invalides")

In [ ]:
# ── CORRECTION Exercice 3 ────────────────────────────────────────────────────

from datetime import datetime

def extraire_mois(date_str):
    """Parse une date 'jj/mm/aaaa' et retourne le numéro du mois."""
    try:
        return datetime.strptime(date_str, "%d/%m/%Y").month
    except ValueError:
        return None

noms_mois = ["Janvier", "Février", "Mars", "Avril", "Mai", "Juin",
             "Juillet", "Août", "Septembre", "Octobre", "Novembre", "Décembre"]

# Construire le CA par mois
ca_par_mois = {}

for cmd, res in zip(commandes, resultats):     # on parcourt commandes et résultats ensemble
    if not res["valide"]:
        continue                                # on saute les commandes invalides
    mois = extraire_mois(cmd["date"])
    if mois is not None:
        ca_par_mois[mois] = ca_par_mois.get(mois, 0) + res["ca_ht"]

# Affichage trié par numéro de mois
for num_mois in sorted(ca_par_mois):
    nom = noms_mois[num_mois - 1]              # index = numéro de mois - 1
    print(f"Mois {num_mois} ({nom:<10}) : {ca_par_mois[num_mois]:>10,.2f} €")

In [ ]:
# ── CORRECTION Exercice 4 ────────────────────────────────────────────────────

def generer_rapport(commandes):
    """Génère un rapport commercial complet à partir d'une liste de commandes."""

    # Traitement de toutes les commandes
    resultats = [traiter_commande(c) for c in commandes]

    # Filtrage des commandes valides
    valides   = [r for r in resultats if r["valide"]]
    invalides = [r for r in resultats if not r["valide"]]

    # Calcul des totaux
    ca_total    = sum(r["ca_ht"] for r in valides)
    marge_total = sum(r["marge"] for r in valides)
    taux_global = calculer_taux_marge(marge_total, ca_total)

    # Trouver le top client (celui avec le plus grand CA cumulé)
    ca_par_client = {}
    for r in valides:
        ca_par_client[r["client"]] = ca_par_client.get(r["client"], 0) + r["ca_ht"]
    top_client = max(ca_par_client, key=lambda x: ca_par_client[x])

    # Affichage du rapport
    L = 42
    print(f"╔{'═' * L}╗")
    print(f"║{'RAPPORT COMMERCIAL T1 2024':^{L}}║")
    print(f"╠{'═' * L}╣")
    print(f"║  {'Commandes traitées :':<22}{len(commandes):>8}          ║")
    print(f"║  {'Commandes valides  :':<22}{len(valides):>8}          ║")
    print(f"║  {'Commandes invalides:':<22}{len(invalides):>8}          ║")
    print(f"║  {'CA total HT        :':<22}{ca_total:>12,.2f} €  ║")
    print(f"║  {'Marge totale       :':<22}{marge_total:>12,.2f} €  ║")
    print(f"║  {'Taux de marge      :':<22}{taux_global:>11.2%}    ║")
    print(f"║  {'Top client         :':<22}{top_client:>14}  ║")
    print(f"╚{'═' * L}╝")

generer_rapport(commandes)